<a href="https://colab.research.google.com/github/sschares/ores5160-2026/blob/main/Schares_ORES5160_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
from urllib.request import urlopen, Request

req = Request("https://data.cdc.gov/resource/8xy9-ubqz.json", headers={'User-Agent': 'Mozilla/5.0'})

with urlopen(req) as response:
  source = response.read()

  print(source)

In [ ]:
data = json.loads(source)
print(type(data))

In [ ]:
print(json.dumps(data, indent = 2))

In [ ]:
import pandas as pd
import numpy as np

telemed = pd.DataFrame(data)

In [ ]:
telemed.head()

In [ ]:
print("Original columns:", telemed.columns.tolist())

# Rename the 'round' column to 'date_range_category'
telemed = telemed.rename(columns={'round': 'date_range_category'})

In [ ]:
print("Updated columns:", telemed.columns.tolist())
display(telemed.head())

In [ ]:
#Drop last 4 columns
telemed = telemed.iloc[:, :-4]

In [ ]:
#Drop sample_size
telemed = telemed.drop(columns=['sample_size'])
display(telemed.head())
telemed.to_csv('telemedicine.csv', index=False)

In [ ]:
telemed_no = telemed[telemed['indicator'] == 'Provider offers telemedicine'][telemed['response'] == 'No'][telemed['subgroup'] == 'Total']
display(telemed_no.head())

In [ ]:
#Recreate telemed_no table with date ranges in place of categories
define_date_range = {
    '1': '2020-06-09 to 2020-07-06',
    '2': '2020-08-03 to 2020-08-20',
    '3': '2021-05-17 to 2021-06-30'}

telemed_no['date_range_category'] = telemed_no['date_range_category'].map(define_date_range)
display(telemed_no.head())


In [ ]:
telemed_yes = telemed[telemed['indicator'] == 'Provider offers telemedicine'][telemed['response'] == 'Yes'][telemed['subgroup'] == 'Total']
display(telemed_yes.head())


In [ ]:
#Recreate telemed_yes table with date ranges in place of categories
define_date_range = {
    '1': '2020-06-09 to 2020-07-06',
    '2': '2020-08-03 to 2020-08-20',
    '3': '2021-05-17 to 2021-06-30'}

telemed_yes['date_range_category'] = telemed_yes['date_range_category'].map(define_date_range)
display(telemed_yes.head())

In [ ]:
telemed.describe(include="all")

In [ ]:
telemed.info()

In [ ]:
telemed = telemed.replace(r'^\s*$', np.nan, regex=True)
telemed.isnull().sum()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Load Google Symptom Search Data
df1 = pd.read_csv('/content/drive/MyDrive/bq-results-20260425-185741-1777143487579/bq-results-20260425-185741-1777143487579.csv')
df1.head()

In [ ]:
#Describe Google Symptom Search Data
df1.info()

In [ ]:
#Convert date in Google Symptom Search Data to datetime
df1['date'] = pd.to_datetime(df1['date'])

In [ ]:
#Describe Google Symptom Search Data
df1.describe(include="all")

In [ ]:
# Add column "Year" to Google Symptom Search Data
df1["year"] = df1["date"].dt.year

In [ ]:
# Create pivot table to calculate Google symptom searches by year
df1.pivot_table(index='year', columns='symptom', values='value', aggfunc='sum')

In [ ]:
from pandas.core import groupby
df1.groupby("symptom")["value"].sum()
orderby = df1.groupby("symptom")["value"].sum().sort_values(ascending=False)
orderby

In [ ]:
plt.figure(figsize=(5, 2.5))
sns.histplot(df1[
    (df1['symptom'] == "symptom_pain")]['value'], bins=10)
plt.title('US Pain Search Distribution ')
plt.xlabel('Relative Number of Searches')
plt.ylabel('Frequency')



In [ ]:
sns.boxplot(df1[
    (df1['symptom'] == "symptom_pain")]['value'])
plt.ylim(0, 100)
plt.title('US Pain Symptom Searches')
plt.ylabel('Relative Frequency of Searches')

In [ ]:
pain_data = df1[(df1['symptom'] == 'symptom_pain')][['date', 'value']].rename(columns={'value': 'Pain_Searches'})
anxiety_data= df1[(df1['symptom'] == 'symptom_anxiety')][['date', 'value']].rename(columns={'value': 'Anxiety_Searches'})

merged_data = pd.merge(pain_data, anxiety_data, on='date', how='inner')

plt.figure(figsize=(8, 4))
sns.scatterplot(data=merged_data, x='Anxiety_Searches', y='Pain_Searches')
plt.title('US Pain vs Anxiety Symptom Searches')
plt.xlabel('Anxiety Searches')
plt.ylabel('Pain Searches')
plt.grid(True)
plt.show()

In [ ]:
import statsmodels.api as sm
pain_data = df1[(df1['symptom'] == 'symptom_pain')][['date', 'value']].rename(columns={'value': 'Pain_Searches'})
anxiety_data= df1[(df1['symptom'] == 'symptom_anxiety')][['date', 'value']].rename(columns={'value': 'Anxiety_Searches'})

merged_data = pd.merge(pain_data, anxiety_data, on='date', how='inner')
sns.regplot(data=merged_data, x='Anxiety_Searches', y='Pain_Searches')
plt.title('US Pain vs Anxiety Symptom Searches')
plt.xlabel('Anxiety Searches')
plt.ylabel('Pain Searches')
plt.grid(True)
plt.show()

In [ ]:
pain_data = df1[(df1['symptom'] == 'symptom_pain')][['date', 'value']].rename(columns={'value': 'Pain_Searches'})
depression_data= df1[(df1['symptom'] == 'symptom_depression')][['date', 'value']].rename(columns={'value': 'Depression_Searches'})

merged_data = pd.merge(pain_data, depression_data, on='date', how='inner')

plt.figure(figsize=(8, 4))
sns.regplot(data=merged_data, x='Depression_Searches', y='Pain_Searches', ci=95)
plt.title('US Pain vs Depression Symptom Searches (95% CI)')
plt.xlabel('Depression Searches')
plt.ylabel('Pain Searches')
plt.grid(True)
plt.show()



In [ ]:
pain_data = df1[(df1['symptom'] == 'symptom_pain')][['date', 'value']].rename(columns={'value': 'Pain_Searches'})
sleep_disorder_data= df1[(df1['symptom'] == 'symptom_sleep_disorder')][['date', 'value']].rename(columns={'value': 'Sleep_Disorder_Searches'})

merged_data = pd.merge(pain_data, sleep_disorder_data, on='date', how='inner')

plt.figure(figsize=(8, 4))
sns.regplot(data=merged_data, x='Sleep_Disorder_Searches', y='Pain_Searches', ci=95)
plt.title('US Pain vs Sleep Disorder Symptom Searches (95% CI)')
plt.xlabel('Sleep Disorder Searches')
plt.ylabel('Pain Searches')
plt.grid(True)
plt.show()

In [ ]:
pain_data = df1[(df1['symptom'] == 'symptom_pain')][['date', 'value']].rename(columns={'value': 'Pain_Searches'})
obesity_data= df1[(df1['symptom'] == 'symptom_obesity')][['date', 'value']].rename(columns={'value': 'Obesity_Searches'})

merged_data = pd.merge(pain_data, obesity_data, on='date', how='inner')

plt.figure(figsize=(8, 4))
sns.regplot(data=merged_data, x='Obesity_Searches', y='Pain_Searches', ci=95)
plt.title('US Pain vs Obesity Symptom Searches (95% CI)')
plt.xlabel('Obesity Searches')
plt.ylabel('Pain Searches')
plt.grid(True)
plt.show()

In [ ]:
symptom_sums = df1[df1['date_range_category'] != 'nan'].groupby(["date_range_category", "symptom"])["value"].sum()
max_searches_by_category = symptom_sums.groupby(level='date_range_category').idxmax()
result = symptom_sums.loc[max_searches_by_category]
display(result)

In [ ]:
#Graph pain searches by category
pain_searches_by_category = df1[df1['symptom'] == "symptom_pain"].groupby('date_range_category')['value'].sum().reset_index()

plt.figure(figsize=(8, 4))
sns.barplot(data=pain_searches_by_category, x='date_range_category', y='value')
plt.title('Pain Searches by Date Range')
plt.xlabel('Date Range')
plt.ylabel('Total Pain Searches Value')

# Define the mapping for date ranges with newlines for wrapping
date_range_labels = {
    '1.0': '2020-06-09\nto 2020-07-06',
    '2.0': '2020-08-03\nto 2020-08-20',
    '3.0': '2021-05-17\nto 2021-06-30'
}

# Convert date_range_category to string for proper mapping
pain_searches_by_category['date_range_category'] = pain_searches_by_category['date_range_category'].astype(str)

# Apply the mapping to the x-axis tick labels, removing rotation for horizontal alignment
plt.xticks(ticks=range(len(pain_searches_by_category['date_range_category'])),
           labels=[date_range_labels.get(cat, cat) for cat in pain_searches_by_category['date_range_category']])

plt.tight_layout() # Adjust layout to prevent labels from being cut off
plt.show()

In [ ]:
#Graph total Google symptom searches by date category
total_searches_by_category = df1.groupby('date_range_category')['value'].sum().reset_index()

plt.figure(figsize=(8, 5))
sns.barplot(data=total_searches_by_category, x='date_range_category', y='value')
plt.title('Total Searches by Date Range')
plt.xlabel('Date Range')
plt.ylabel('Total Searches Value')

# Define the mapping for date ranges with newlines for wrapping
date_range_labels = {
    '1.0': '2020-06-09\nto 2020-07-06',
    '2.0': '2020-08-03\nto 2020-08-20',
    '3.0': '2021-05-17\nto 2021-06-30'
}
# Convert date_range_category to string for proper mapping
total_searches_by_category['date_range_category'] = total_searches_by_category['date_range_category'].astype(str)

# Apply the mapping to the x-axis tick labels, removing rotation for horizontal alignment
plt.xticks(ticks=range(len(total_searches_by_category['date_range_category'])),
           labels=[date_range_labels.get(cat, cat) for cat in total_searches_by_category['date_range_category']])

plt.tight_layout() # Adjust layout to prevent labels from being cut off
plt.show()

In [ ]:
df2 = pd.read_csv('/content/drive/MyDrive/bq-results-20260425-132419-1777124125607/bq-results-20260425-132419-1777124125607.csv')
df2.head()

In [ ]:
df2.info()

In [ ]:
df2.describe(include="all")

In [ ]:
df2 = df2.replace(r'^\s*$', np.nan, regex=True)
df2.isnull().sum()

In [ ]:
telemed_yes['percent'] = pd.to_numeric(telemed_yes['percent'], errors='coerce')

plt.figure(figsize=(8, 5))
sns.barplot(data=telemed_yes, x='date_range_category', y='percent')
plt.title('Telemedicine = "Yes" by Date Range')
plt.xlabel('Date Range')
plt.ylabel('Percent Telemedicine = Yes')

plt.tight_layout() # Adjust layout to prevent labels from being cut off

In [ ]:
telemed_no_data = telemed[
    (telemed['indicator'] == 'Provider offers telemedicine') &
    (telemed['response'] == 'No') &
    (telemed['subgroup'] == 'Total')
].copy()

telemed_no_data['percent'] = pd.to_numeric(telemed_no_data['percent'], errors='coerce')
telemed_no_data['date_range_category'] = pd.to_numeric(telemed_no_data['date_range_category'])

pain_searches_by_category = df2[
    (df2['symptom'] == 'symptom_pain') &
    (df2['indicator'] == 'Provider offers telemedicine') &
    (df2['subgroup'] == 'Total') &
    (df2['response'] == 'No')
].groupby('date_range_category')['value'].sum().reset_index()

merged_data_no = pd.merge(telemed_no_data, pain_searches_by_category, on='date_range_category', how='inner')
display(merged_data_no.head())

In [ ]:
plt.figure(figsize=(8, 6))
sns.regplot(data=merged_data_no, x='percent', y='value', ci=95)
plt.title('Relationship Between Telemedicine "No" Percent and Pain Searches')
plt.xlabel('Percent of Providers NOT Offering Telemedicine')
plt.ylabel('Total Pain Searches Value')
plt.grid(True)
plt.show()

In [ ]:
telemed_yes_data = telemed[
    (telemed['indicator'] == 'Provider offers telemedicine') &
    (telemed['response'] == 'Yes') &
    (telemed['subgroup'] == 'Total')
].copy()

telemed_yes_data['percent'] = pd.to_numeric(telemed_yes_data['percent'], errors='coerce')
telemed_yes_data['date_range_category'] = pd.to_numeric(telemed_yes_data['date_range_category'])

pain_searches_by_category = df2[
    (df2['symptom'] == 'symptom_pain') &
    (df2['indicator'] == 'Provider offers telemedicine') &
    (df2['subgroup'] == 'Total') &
    (df2['response'] == 'Yes')
].groupby('date_range_category')['value'].sum().reset_index()

merged_data_yes = pd.merge(telemed_yes_data, pain_searches_by_category, on='date_range_category', how='inner')
display(merged_data_yes.head())

In [ ]:
plt.figure(figsize=(8, 6))
sns.regplot(data=merged_data_yes, x='percent', y='value', ci=95)
plt.title('Relationship Between "Telemedicine = Yes" Percent and Pain Searches')
plt.xlabel('Percent of Providers Offering Telemedicine (Yes)')
plt.ylabel('Total Pain Searches Value')
plt.grid(True)
plt.show()